# AH | Pruebas de reglas

**Autor:** Alan Arturo Hernández Gutiérrez.

Comprueba casos sintéticos de identidad, seguimiento, preferencias, centavos e indicadores municipales. Los datos de prueba se usan exclusivamente en las comprobaciones.

Este cuaderno define las operaciones que utilizan los cuadernos de `notebooks/`. Ejecute sus celdas en orden. El flujo completo se inicia desde 01 o 02.

## Raíz del proyecto

La ubicación se resuelve desde la raíz, `src/` o `notebooks/`, sin guardar rutas personales.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name.lower() in {'src', 'notebooks'}:
    ROOT = ROOT.parent
if not (ROOT / 'config' / 'proyecto.json').is_file():
    raise RuntimeError('Inicie Jupyter desde la raíz del proyecto, src/ o notebooks/.')


## Funciones del modelo

Las pruebas utilizan las mismas funciones que ejecuta el ETL.

In [ ]:
if 'snapshot' not in globals():
    cuaderno_dependencia = ROOT / 'src' / 'etl.ipynb'
    %run "$cuaderno_dependencia"


## Bibliotecas y parámetros

Se preparan las bibliotecas y constantes utilizadas por las operaciones siguientes.

In [ ]:
from io import BytesIO

from pathlib import Path

import tempfile

import unittest

import zipfile

import pandas as pd


## Preparar los casos sintéticos

Crea eventos de prueba con compras físicas y digitales, empates y seguimiento incompleto. Este conjunto se utiliza en las pruebas de reglas.

In [ ]:
def sample_events():
    rows = [
        (1, "2020-01-01", "Fisica", 1000),
        (1, "2020-01-01", "Fisica", 500),
        (1, "2020-01-31", "Internet", 2000),
        (1, "2020-03-01", "Fisica", 3000),
        (2, "2020-01-01", "Fisica", 1000),
        (3, "2020-03-01", "Fisica", 1000),
        (4, "2020-01-01", "Fisica", 1000),
        (4, "2020-01-01", "Internet", 1000),
        (5, "2020-01-01", "Desconocido", 1000),
        (5, "2020-02-01", "Fisica", 1000),
    ]
    d = pd.DataFrame(rows, columns=["ClienteKey", "Fecha", "Canal", "MontoCentavos"])
    d.Fecha = pd.to_datetime(d.Fecha)
    d["LineaFuenteNro"] = range(1, len(d) + 1)
    d["TicketKey"] = range(1, len(d) + 1)
    d["TiendaKey"] = 1
    d["CategoriaCodigo"] = "1"
    d["Unidades"] = 1.0
    return d


## Comprobar doce reglas de frontera

Cada método prueba una condición concreta: fechas, madurez, antecedentes, canal, preferencias, centavos o nivel municipal ITER. Los fallos identifican la regla afectada.

In [ ]:
class ReglasETL(unittest.TestCase):

    def setUp(self):
        self.s = snapshot(sample_events(), pd.Timestamp("2020-04-01"), 1).set_index("ClienteKey")

    def test_visita_no_es_linea(self):
        self.assertEqual(int(self.s.loc[1, "VisitasAntecedentes"]), 1)
        self.assertEqual(int(self.s.loc[1, "LineasAntecedentes"]), 2)

    def test_antecedentes_no_usar_futuro(self):
        self.assertEqual(int(self.s.loc[1, "CentavosAntecedentes"]), 1500)
        self.assertEqual(int(self.s.loc[1, "InactividadReferenciaDias"]), 30)

    def test_horizonte_inclusivo_30_dias(self):
        self.assertEqual(int(self.s.loc[1, "Adopto30"]), 1)

    def test_madurez_nulo_no_cero(self):
        self.assertTrue(pd.isna(self.s.loc[3, "Adopto90"]))
        self.assertTrue(pd.isna(self.s.loc[3, "Recompra90"]))
        self.assertEqual(int(self.s.loc[2, "Adopto90"]), 0)

    def test_empate_canal_no_inventar_orden(self):
        self.assertEqual(self.s.loc[4, "EstadoOrigen"], "Empate fisico-internet")
        self.assertTrue(pd.isna(self.s.loc[4, "Adopto90"]))

    def test_desconocido_no_convertir_a_fisico(self):
        self.assertEqual(self.s.loc[5, "EstadoOrigen"], "Origen desconocido")
        self.assertEqual(int(self.s.loc[5, "EsElegibleFisico"]), 0)

    def test_corte_anterior_no_usa_digital_futuro(self):
        s = snapshot(sample_events(), pd.Timestamp("2020-01-20"), 1).set_index("ClienteKey")
        self.assertEqual(int(s.loc[1, "EsAdoptantePosterior"]), 0)
        self.assertEqual(int(s.loc[1, "CentavosPeriodo"]), 1500)

    def test_preferencia_desempate_importe_y_unidades(self):
        d = pd.DataFrame(
            {
                "ClienteKey": [1, 1, 1],
                "TiendaKey": [3, 2, 1],
                "Fecha": pd.to_datetime(["2020-01-01"] * 3),
                "MontoCentavos": [100, 200, 200],
                "Unidades": [1, 1, 2],
            }
        )
        best = preferred(d, "TiendaKey").iloc[0]
        self.assertEqual(int(best.TiendaKey), 1)
        self.assertEqual(int(best.EmpatePreferencia), 0)

    def test_preferencia_empate_tecnico_estable(self):
        d = pd.DataFrame(
            {
                "ClienteKey": [1, 1],
                "TiendaKey": [7, 3],
                "Fecha": pd.to_datetime(["2020-01-01"] * 2),
                "MontoCentavos": [200, 200],
                "Unidades": [2, 2],
            }
        )
        best = preferred(d, "TiendaKey").iloc[0]
        self.assertEqual(int(best.TiendaKey), 3)
        self.assertEqual(int(best.EmpatePreferencia), 1)

    def test_preserva_signos_y_centavos(self):
        self.assertEqual(
            list(money_cents(pd.Series([-1.23, 0, 0.01, 199793.69]))), [-123, 0, 1, 19979369]
        )

    def test_no_redondea_precision_desconocida(self):
        with self.assertRaises(ValueError):
            money_cents(pd.Series([1.234]))

    def test_iter_no_suma_localidades_y_totales(self):
        csv = "ENTIDAD,NOM_ENT,MUN,NOM_MUN,LOC,POBTOT,VIVPARH_CV,VPH_INTER\n01,Aguascalientes,000,Total,0000,1000,400,200\n01,Aguascalientes,001,Aguascalientes,0000,800,300,150\n01,Aguascalientes,001,Aguascalientes,0001,400,100,90\n01,Aguascalientes,002,Asientos,0000,200,100,*\n"
        with tempfile.TemporaryDirectory(prefix="test_iter_") as tmp:
            p = Path(tmp) / "prueba_sintetica.zip"
            with zipfile.ZipFile(p, "w") as z:
                z.writestr("conjunto_de_datos/datos.csv", csv)
            d, meta = cargar_iter_municipal(p)
        self.assertEqual(len(d), 2)
        self.assertEqual(int(d.PoblacionMunicipio2020.sum()), 1000)
        self.assertEqual(float(d.iloc[0].PorcentajeInternetMunicipio2020), 0.5)
        self.assertTrue(pd.isna(d.iloc[1].PorcentajeInternetMunicipio2020))


## Run tests

Ejecuta las pruebas y devuelve conteos y detalle; cualquier fallo interrumpe el ETL.

In [ ]:
def run_tests():
    suite = unittest.defaultTestLoader.loadTestsFromTestCase(ReglasETL)
    stream = __import__("io").StringIO()
    result = unittest.TextTestRunner(stream=stream, verbosity=2).run(suite)
    if not result.wasSuccessful():
        raise AssertionError(stream.getvalue())
    return {
        "pruebas": result.testsRun,
        "fallos": len(result.failures),
        "errores": len(result.errors),
        "detalle": stream.getvalue(),
    }
